# 05 — Table Order Reviews

Nettoyage de la table `order_reviews` : normalisation du texte des commentaires et options de remplacement des commentaires vides (choix à faire par le groupe).

*Extrait de `branche-emma` (commit `9de2053`), autrice : Emma. La cellule de chargement est celle commune aux autres notebooks.*

On commence par importer les librairies nécessaires et charger les fichiers CSV dans un dictionnaire de DataFrames. Ensuite, nous créons une copie du dictionnaire original pour conserver les données brutes.

In [1]:
# Imports des bibliothèques

import glob

import pandas as pd

# afficher toutes les colonnes
pd.set_option("display.max_columns", 50)

all_files = sorted(glob.glob("*.csv"))

df_dic = {}
df_origin = {}
for file in all_files:
    nom = file.replace("datasets/Olist/", "").replace("olist_", "").replace("_dataset", "").replace(".csv", "")
    df_dic[nom] = pd.read_csv(file)
    df_origin[nom] = pd.read_csv(file)


def resume(df):
    """Renvoie un tableau de synthèse: une ligne par colonne  de df."""
    return pd.DataFrame(
        {
            "type": df.dtypes,
            "nb_uniques": df.nunique(),
            "maximum": df.max(),
            "minimum": df.min(),
            "nb manquants": df.isnull().sum(),
            "pct_manquants": (df.isnull().mean() * 100).round(2),
        }
    )

## Table Reviews

In [2]:
reviews = df_dic["order_reviews"]
resume(reviews)

,type,nb_uniques,maximum,minimum,nb manquants,pct_manquants
review_id,str,98410,fffefe7a48d22f7b32046421062219d1,0001239bc1de2e33cb583967c2ca4c67,0,0.00
order_id,str,98673,fffe41c64501cc87c801fd61db3f6244,00010242fe8c5a6d1ba2dd792cb16214,0,0.00
review_score,int64,5,5,1,0,0.00
review_comment_title,str,4527,🔟,,87656,88.34
review_comment_message,str,36159,😡😡😡😡😡👎👎👎👎👎\r\nEmpresa sem compromisso com o cl...,\r\n,58247,58.70
review_creation_date,str,636,2018-08-31 00:00:00,2016-10-02 00:00:00,0,0.00
review_answer_timestamp,str,98248,2018-10-29 12:27:35,2016-10-07 18:32:28,0,0.00


### Nettoyage des colonnes review_comment_title et review_comment_message

In [3]:
# passage en minuscule 
reviews["review_comment_message"] = reviews["review_comment_message"].str.lower()
reviews["review_comment_title"] = reviews["review_comment_title"].str.lower()

# enlever les emojis
import re 
reviews['review_comment_message'] = reviews['review_comment_message'].apply(lambda x: re.sub(r'[^\w\s.,!?;:\'\"()\-À-ÿ]', '', x) if isinstance(x, str) else x)
reviews['review_comment_title'] = reviews['review_comment_title'].apply(lambda x: re.sub(r'[^\w\s.,!?;:\'\"()\-À-ÿ]', '', x) if isinstance(x, str) else x)

# enlever les retours à la ligne 
reviews["review_comment_message"] = reviews["review_comment_message"].str.replace(r"[\r\n]+", " ", regex=True).str.strip()
reviews["review_comment_title"] = reviews["review_comment_title"].str.replace(r"[\r\n]+", " ", regex=True).str.strip()


In [4]:
# récupère les 50 premiers messages/titres qui ne sont pas NaN et dont la note est égale à note, et les stocke dans la liste 
def recuperer_reviews_by_note(note):

    liste_reviews_message = []
    liste_reviews_title = []

    compteur_lignes = 0 

    while len(liste_reviews_message) < 50 and len(liste_reviews_title) < 50 and compteur_lignes < len(reviews) : 

        message = reviews['review_comment_message'].iloc[compteur_lignes]
        title = reviews['review_comment_title'].iloc[compteur_lignes]

        if pd.notna(message) and reviews['review_score'].iloc[compteur_lignes] == note :
            liste_reviews_message.append(message)

        if pd.notna(title) and reviews['review_score'].iloc[compteur_lignes] == note :
            liste_reviews_title.append(title)

        compteur_lignes += 1

    return liste_reviews_message, liste_reviews_title


random_message_note_5, random_title_note_5 = recuperer_reviews_by_note(5) 
random_message_note_4, random_title_note_4 = recuperer_reviews_by_note(4) 
random_message_note_3, random_title_note_3 = recuperer_reviews_by_note(3) 
random_message_note_2, random_title_note_2 = recuperer_reviews_by_note(2) 
random_message_note_1, random_title_note_1 = recuperer_reviews_by_note(1) 

#est-ce qu'on l'utilise ou pas au final ? 

In [5]:
#remplir les cases vides avec les messages/titles random en fonction des notes 
import random 

def remplacer_NaN_par_random(liste_random_message, liste_random_title): 
     
    reviews["review_comment_message"] = reviews["review_comment_message"].apply(
        lambda x: random.choice(liste_random_message) if pd.isna(x) else x
    )

    reviews["review_comment_title"] = reviews["review_comment_title"].apply(
        lambda x: random.choice(liste_random_title) if pd.isna(x) else x
    )

# PAS UTILISÉ, on en fait quoi ? si on l'utilise alors décommenter les cinq lignes ci-dessous 
# remplacer_NaN_par_random(random_message_note_5, random_title_note_5)
# remplacer_NaN_par_random(random_message_note_4, random_title_note_4)
# remplacer_NaN_par_random(random_message_note_3, random_title_note_3)
# remplacer_NaN_par_random(random_message_note_2, random_title_note_2)
# remplacer_NaN_par_random(random_message_note_1, random_title_note_1)


In [6]:
# remplacer tous les NaN par la valeur None 
def remplacer_NaN_par_None(): 
    reviews["review_comment_message"] = reviews["review_comment_message"].fillna("None")
    reviews["review_comment_title"] = reviews["review_comment_title"].fillna("None") 

# PAS UTILISÉ NON PLUS, décommenter ci-dessous si on choisit cette option 
# remplacer_NaN_par_None()